# 📘 Week 04 · Day 20 | PM — Normal Distribution, Z-Score & Hypothesis Testing
**Topics:** Normal Distribution, Standard Normal, Z-score, Hypothesis Testing, Descriptive Statistics

**Deadline:** 19/03/2026 · 09:15 AM

---

## ⚙️ Setup

In [ ]:
!pip install numpy matplotlib scipy -q

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import math

np.random.seed(42)
plt.rcParams["figure.figsize"]  = (8, 4)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
print("Ready.")

---
## 🧠 Part A — Concept Application

### A1 — Generate Normal(50, 10) dataset (n=1000) — manual stats + histogram

In [ ]:
n_points = 1000
pop_mean = 50
pop_std  = 10

data = list(np.random.normal(loc=pop_mean, scale=pop_std, size=n_points))

# ── Manual mean ────────────────────────────────────────────────────────────
def manual_mean(lst):
    total = 0
    count = 0
    for x in lst:
        total = total + x
        count = count + 1
    return total / count

# ── Manual variance (population) ──────────────────────────────────────────
def manual_variance(lst):
    m = manual_mean(lst)
    sq_sum = 0
    for x in lst:
        sq_sum = sq_sum + (x - m) ** 2
    count = 0
    for _ in lst:
        count += 1
    return sq_sum / count

# ── Manual std ────────────────────────────────────────────────────────────
def manual_std(lst):
    var   = manual_variance(lst)
    guess = var / 2.0
    for _ in range(50):          # Newton's method for sqrt
        guess = (guess + var / guess) / 2.0
    return guess

m   = manual_mean(data)
v   = manual_variance(data)
s   = manual_std(data)

print(f"Generated {n_points} samples from Normal(mean={pop_mean}, std={pop_std})")
print()
print(f"Manual mean:     {m:.4f}   (expected ~{pop_mean})")
print(f"Manual variance: {v:.4f}   (expected ~{pop_std**2})")
print(f"Manual std:      {s:.4f}   (expected ~{pop_std})")
print()
print(f"NumPy mean:     {np.mean(data):.4f}")
print(f"NumPy variance: {np.var(data):.4f}")
print(f"NumPy std:      {np.std(data):.4f}")

fig, ax = plt.subplots()
ax.hist(data, bins=40, density=True, color="steelblue", alpha=0.7,
        edgecolor="white", label="Histogram")
x_r = np.linspace(min(data), max(data), 300)
ax.plot(x_r, stats.norm.pdf(x_r, pop_mean, pop_std),
        color="red", lw=2, label=f"Theoretical Normal({pop_mean},{pop_std})")
ax.axvline(m, color="orange", linestyle="--", lw=1.5, label=f"Mean={m:.1f}")
ax.set_title("Normal Distribution — n=1000")
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()
print("Observation: bell-shaped, symmetric around the mean. Bars fit the red curve closely.")

### A2 — Z-score (standardisation) — manual implementation

Z = (x − mean) / std.  After transformation: mean ≈ 0, std ≈ 1.

In [ ]:
def compute_zscore(lst):
    """
    Converts every value in lst to its Z-score.
    Z = (x - mean) / std
    Returns a new list of standardised values.
    """
    m = manual_mean(lst)
    s = manual_std(lst)
    z_vals = []
    for x in lst:
        z_vals.append((x - m) / s)
    return z_vals


z_data = compute_zscore(data)

z_mean = manual_mean(z_data)
z_std  = manual_std(z_data)

print(f"After Z-score transformation:")
print(f"  Mean of Z-scores: {z_mean:.8f}   (expected ~0)")
print(f"  Std  of Z-scores: {z_std:.8f}   (expected ~1)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(data,   bins=40, density=True, color="steelblue", alpha=0.7)
axes[0].set_title(f"Original  — mean={manual_mean(data):.1f}, std={manual_std(data):.1f}")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Density")

axes[1].hist(z_data, bins=40, density=True, color="darkorange", alpha=0.7)
axes[1].set_title(f"Standardised Z-score  — mean≈0, std≈1")
axes[1].set_xlabel("Z-score")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

### A3 — Student marks: mean, median, variance, std + outlier detection via Z-score

In [ ]:
marks = [
    55, 62, 70, 45, 88, 92, 35, 78, 65, 73,
    81, 58, 90, 48, 77, 69, 84, 53, 96,  2,
    67, 74, 61, 99, 57, 83, 71, 49, 76, 105
]

# ── Manual median ──────────────────────────────────────────────────────────
def manual_median(lst):
    s = lst[:]
    n = 0
    for _ in s:
        n += 1
    # bubble sort
    for i in range(n):
        for j in range(0, n - i - 1):
            if s[j] > s[j + 1]:
                s[j], s[j + 1] = s[j + 1], s[j]
    mid = n // 2
    if n % 2 == 0:
        return (s[mid - 1] + s[mid]) / 2
    return s[mid]

m_marks  = manual_mean(marks)
md_marks = manual_median(marks)
v_marks  = manual_variance(marks)
s_marks  = manual_std(marks)

print(f"Student marks statistics:")
print(f"  Mean:     {m_marks:.4f}")
print(f"  Median:   {md_marks:.4f}")
print(f"  Variance: {v_marks:.4f}")
print(f"  Std Dev:  {s_marks:.4f}")
print()

# Outlier detection: |Z| > 2 (broad) and |Z| > 3 (strict)
z_marks = compute_zscore(marks)

outliers_2 = [(marks[i], round(z_marks[i], 3)) for i in range(len(marks))
              if abs(z_marks[i]) > 2]
outliers_3 = [(marks[i], round(z_marks[i], 3)) for i in range(len(marks))
              if abs(z_marks[i]) > 3]

print(f"Outliers |Z| > 2  (unusual):  {outliers_2}")
print(f"Outliers |Z| > 3  (extreme):  {outliers_3}")

fig, ax = plt.subplots()
ax.hist(marks, bins=15, color="steelblue", alpha=0.7, edgecolor="white")
ax.axvline(m_marks,  color="red",   linestyle="--", lw=2, label=f"Mean={m_marks:.1f}")
ax.axvline(md_marks, color="green", linestyle="--", lw=2, label=f"Median={md_marks:.1f}")
for val, _ in outliers_2:
    ax.axvline(val, color="orange", linestyle=":", lw=1.5, alpha=0.7)
ax.set_title("Student Marks — Outliers marked in orange dotted lines")
ax.set_xlabel("Marks")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

### A4 — One-sample hypothesis test (Z-statistic, manual, α = 0.05)

In [ ]:
def one_sample_z_test(sample, null_mean, alpha=0.05):
    """
    Tests H0: population mean = null_mean
         H1: population mean != null_mean  (two-tailed)

    Steps:
      1. Compute sample mean and std
      2. Z = (sample_mean - null_mean) / (std / sqrt(n))
      3. Critical value for two-tailed test at alpha=0.05 is ±1.96
      4. Reject H0 if |Z| > 1.96
    """
    n           = 0
    for _ in sample:
        n += 1
    sample_mean = manual_mean(sample)
    sample_std  = manual_std(sample)

    # standard error of the mean
    se     = sample_std / (n ** 0.5)
    z_stat = (sample_mean - null_mean) / se

    # critical value for two-tailed test at alpha=0.05
    # z_critical = 1.96 (from standard normal table)
    z_critical = 1.96

    # p-value: area in both tails beyond |z_stat|
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    reject = abs(z_stat) > z_critical

    print("=" * 52)
    print("ONE-SAMPLE Z-TEST")
    print("=" * 52)
    print(f"  H0: population mean = {null_mean}")
    print(f"  H1: population mean != {null_mean}  (two-tailed)")
    print(f"  Sample size:    {n}")
    print(f"  Sample mean:    {sample_mean:.4f}")
    print(f"  Sample std:     {sample_std:.4f}")
    print(f"  Z-statistic:    {z_stat:.4f}")
    print(f"  Critical value: ±{z_critical}")
    print(f"  p-value:        {p_value:.6f}")
    print(f"  Alpha:          {alpha}")
    print()
    if reject:
        print(f"  REJECT H0: The sample mean is significantly different from {null_mean}")
    else:
        print(f"  FAIL TO REJECT H0: No significant evidence against mean = {null_mean}")
    print("=" * 52)
    return z_stat, p_value, reject


# Test on our generated dataset — null mean = 50 (true mean, should not reject)
print("--- Test 1: null mean = 50 (same as true mean) ---")
z1, p1, r1 = one_sample_z_test(data, null_mean=50)

print()
# Test on student marks — null mean = 65 (plausible)
print("--- Test 2: student marks, null mean = 65 ---")
z2, p2, r2 = one_sample_z_test(marks, null_mean=65)

### A5 — Simulate 1000 tests under H₀ true — estimate false positive rate

In [ ]:
# H0 is TRUE: we draw from Normal(50, 10) and test H0: mean = 50
# At alpha=0.05 we expect roughly 5% false positives (Type I errors)

n_simulations  = 1000
n_per_sample   = 30
true_mean      = 50
true_std       = 10
alpha          = 0.05
z_critical     = 1.96

false_positives = 0

for _ in range(n_simulations):
    sample = np.random.normal(true_mean, true_std, n_per_sample)
    se     = np.std(sample) / (n_per_sample ** 0.5)
    z      = (np.mean(sample) - true_mean) / se
    if abs(z) > z_critical:
        false_positives += 1

fp_rate = false_positives / n_simulations

print(f"Simulations:         {n_simulations}")
print(f"n per sample:        {n_per_sample}")
print(f"True H0 mean:        {true_mean}")
print(f"Alpha:               {alpha}")
print()
print(f"False positives:     {false_positives}")
print(f"False positive rate: {fp_rate:.4f}  ({fp_rate*100:.2f}%)")
print(f"Expected (alpha):    {alpha:.4f}  ({alpha*100:.2f}%)")
print()
diff = abs(fp_rate - alpha)
print(f"Difference from alpha: {diff:.4f}")
print("Interpretation: The simulated false positive rate is close to alpha = 0.05.")
print("This confirms that when H0 is TRUE, we still incorrectly reject it ~5% of the time.")
print("That 5% is the Type I error we accepted by choosing alpha = 0.05.")

---
## 🚀 Part B — Stretch

### B1 — Normal vs Standard Normal: generate, plot, explain differences

In [ ]:
normal_samples   = np.random.normal(loc=70, scale=15, size=2000)
standard_samples = (normal_samples - np.mean(normal_samples)) / np.std(normal_samples)

print("Normal(70, 15):")
print(f"  Mean={np.mean(normal_samples):.2f}  Std={np.std(normal_samples):.2f}  Range=[{np.min(normal_samples):.1f}, {np.max(normal_samples):.1f}]")
print("Standard Normal (Z-scored):")
print(f"  Mean={np.mean(standard_samples):.8f}  Std={np.std(standard_samples):.8f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_n = np.linspace(normal_samples.min(), normal_samples.max(), 300)
axes[0].hist(normal_samples, bins=50, density=True, color="steelblue", alpha=0.7)
axes[0].plot(x_n, stats.norm.pdf(x_n, 70, 15), color="red", lw=2)
axes[0].set_title("Normal(70, 15)  — Original Scale")
axes[0].set_xlabel("Marks")
axes[0].set_ylabel("Density")

x_z = np.linspace(-4, 4, 300)
axes[1].hist(standard_samples, bins=50, density=True, color="darkorange", alpha=0.7)
axes[1].plot(x_z, stats.norm.pdf(x_z, 0, 1), color="red", lw=2)
axes[1].set_title("Standard Normal N(0,1)  — Z-score Scale")
axes[1].set_xlabel("Z-score (standard deviations from mean)")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
print("""
Normal vs Standard Normal:

Normal Distribution:
  Can have ANY mean and ANY standard deviation.
  Units stay in the original scale (marks, rupees, minutes).
  Each different Normal is a separate distribution.

Standard Normal (Z-distribution):
  Always has mean = 0 and standard deviation = 1.
  Units are 'standard deviations from the mean' (dimensionless).
  All Normal distributions collapse into this one after standardisation.

Key differences in the plots:
  Shape:  Both are identical bell curves.
  X-axis: Normal shows original values (e.g., 40 to 100 marks).
          Standard Normal shows Z-scores (-3 to +3 std deviations).
  Y-axis: Density changes scale but area under both = 1.

Why standardise:
  To compare values from different distributions.
  Example: a score of 80 in one exam and 75 in another exam are
  not directly comparable -- but Z = +1.5 in both means the same
  thing: 1.5 std deviations above average.
""")

### B2 — Hypothesis testing on two groups: compare means

In [ ]:
# Group 1: old teaching method
# Group 2: new teaching method
np.random.seed(10)
group_old = list(np.random.normal(65, 12, 40).round(1))
group_new = list(np.random.normal(72, 11, 40).round(1))

mean_old  = manual_mean(group_old)
mean_new  = manual_mean(group_new)
diff      = mean_new - mean_old

print(f"Group OLD  — n=40  mean={mean_old:.2f}  std={manual_std(group_old):.2f}")
print(f"Group NEW  — n=40  mean={mean_new:.2f}  std={manual_std(group_new):.2f}")
print(f"Difference in means (NEW - OLD): {diff:.2f}")

# Basic t-test via scipy for verification
t_stat, p_val = stats.ttest_ind(group_new, group_old)
print()
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_val:.6f}")
if p_val < 0.05:
    print("Conclusion: The new teaching method produces SIGNIFICANTLY higher marks (p < 0.05).")
else:
    print("Conclusion: No significant difference detected (p >= 0.05).")

fig, ax = plt.subplots()
ax.hist(group_old, bins=15, alpha=0.6, color="steelblue",  label=f"Old (mean={mean_old:.1f})")
ax.hist(group_new, bins=15, alpha=0.6, color="darkorange", label=f"New (mean={mean_new:.1f})")
ax.axvline(mean_old, color="steelblue",  linestyle="--", lw=2)
ax.axvline(mean_new, color="darkorange", linestyle="--", lw=2)
ax.set_title("Old vs New Teaching Method — Score Distributions")
ax.set_xlabel("Score")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

### B3 — When to standardise and why Z-score matters in ML

In [ ]:
print("""
WHEN SHOULD YOU STANDARDISE DATA?

1. Distance-based algorithms:
   KNN, K-Means, SVM, PCA all compute distances between data points.
   If one feature is in thousands (income) and another is 0-1 (age/100),
   the large-scale feature dominates. Standardising puts all features
   on the same scale so each contributes equally.

2. Gradient descent (Linear/Logistic Regression, Neural Networks):
   Un-normalised features cause elongated loss surfaces. Gradient descent
   takes tiny steps in the steep direction and huge steps in the flat one,
   making convergence slow and erratic.
   Standardised features create circular contours -- faster convergence.

3. Regularisation (L1/L2 penalties):
   The penalty treats all weights equally. If features have different scales,
   the penalty unfairly punishes the weights of large-scale features.
   Standardise first so regularisation is applied fairly.

4. Neural network weight initialisation and batch normalisation:
   Inputs near zero mean and unit variance prevent vanishing/exploding
   gradients during training.

WHY Z-SCORE IS IMPORTANT IN ML:

  The Z-score formula: Z = (x - mean) / std

  1. Interpretability: Z = +2 means '2 standard deviations above average'.
     This is comparable across features and datasets.

  2. Outlier detection: |Z| > 3 flags extreme values consistently,
     independent of the original scale.

  3. Statistical tests: Z-tests and t-tests assume standardised inputs.
     Computing a p-value requires knowing how many std deviations the
     observed difference is from the null hypothesis.

  4. Feature engineering: Standardised features are directly comparable
     in feature importance plots and coefficient tables.

When NOT to standardise:
  - Tree-based models (Random Forest, XGBoost): splits are on raw values;
    scaling has no effect on the model or accuracy.
  - When the original scale has meaning (e.g., money, where the range matters).
""")

---
## 💼 Part C — Interview Ready

### Q1 — Normal vs Standard Normal Distribution

In [ ]:
print("""
Normal Distribution N(mu, sigma^2):
  A family of continuous bell-shaped distributions.
  Parameterised by mean (mu) and standard deviation (sigma).
  mu controls the centre; sigma controls the width.
  Example: exam marks ~ Normal(65, 12^2), income ~ Normal(50000, 20000^2).

Standard Normal Distribution N(0, 1):
  The SPECIFIC Normal distribution where mu=0 and sigma=1.
  Created by Z-scoring any Normal: Z = (X - mu) / sigma.
  Every Normal distribution becomes Standard Normal after standardisation.
  The Z-table (or scipy.stats.norm.cdf) always refers to N(0,1).

Relationship:
  If X ~ Normal(mu, sigma^2), then Z = (X - mu)/sigma ~ N(0, 1).

Practical difference:
  Normal: tells you values in meaningful units (marks, money, time).
  Standard Normal: tells you HOW MANY standard deviations a value is
  from the mean -- a universal, unit-free measure.

  P(X > 80) for Normal(65, 12) requires looking up P(Z > (80-65)/12) = P(Z > 1.25).
  That last step uses the Standard Normal table.

When to use each:
  Normal: model raw data; generate samples; describe a real-world quantity.
  Standard Normal: probability lookups; hypothesis tests; comparing values
  from different distributions; Z-score outlier detection.
""")

### Q2 — `z_score()` function applied to a dataset

In [ ]:
def z_score(x, mean, std):
    """
    Returns the standardised (Z-score) value of x.
    Z = (x - mean) / std
    Tells you how many standard deviations x is from the mean.
    """
    return (x - mean) / std


# Apply on student marks dataset
mark_vals = [55, 62, 70, 45, 88, 92, 35, 78, 65, 73,
             81, 58, 90, 48, 77, 69, 84, 53, 96,  2]

mean_val = manual_mean(mark_vals)
std_val  = manual_std(mark_vals)

print(f"Dataset mean: {mean_val:.2f}   std: {std_val:.2f}")
print()
print(f"{'Mark':>5}  {'Z-score':>8}  {'Interpretation'}")
print("-" * 45)
for mark in mark_vals:
    z = z_score(mark, mean_val, std_val)
    if z >  2:  interp = "very high outlier"
    elif z >  1:  interp = "above average"
    elif z > -1:  interp = "near average"
    elif z > -2:  interp = "below average"
    else:         interp = "very low outlier"
    print(f"  {mark:3d}    {z:+.4f}    {interp}")

### Q3 — Hypothesis Testing: full explanation with all four concepts

In [ ]:
print("""
HYPOTHESIS TESTING:

NULL HYPOTHESIS (H0):
  The default assumption -- 'nothing changed', 'no difference', 'no effect'.
  You assume H0 is true and try to find evidence strong enough to reject it.
  H0 is conservative: you need strong evidence to overturn it.
  Example: 'The new drug has no effect on blood pressure.'
  Example: 'The mean exam score is still 65.'

ALTERNATIVE HYPOTHESIS (H1 or Ha):
  The claim you want to demonstrate.
  It is what you believe might be true if H0 is wrong.
  Example: 'The new drug DOES lower blood pressure.'
  Two-tailed: H1: mean != 65
  One-tailed: H1: mean > 65 or H1: mean < 65

P-VALUE:
  The probability of observing data AS EXTREME as yours,
  ASSUMING H0 is completely true.
  Small p-value: your data would be very rare under H0 -- evidence against H0.
  Large p-value: your data is plausible under H0 -- not enough evidence to reject.
  Common mistake: p-value is NOT P(H0 is true). It is P(data | H0 true).

SIGNIFICANCE LEVEL (alpha):
  The threshold you set BEFORE the test for how small p must be to reject H0.
  Typically alpha = 0.05 (5%).
  Meaning: 'I accept a 5% chance of wrongly rejecting H0 (Type I error).'
  More cautious tests (medical decisions) use alpha = 0.01 or 0.001.

Decision rule:
  p < alpha  -->  REJECT H0  (result is 'statistically significant')
  p >= alpha -->  FAIL TO REJECT H0  (not enough evidence)

Worked example:
  H0: class mean = 65.  You collect 30 marks, get mean = 72, std = 10.
  Z = (72 - 65) / (10 / sqrt(30)) = 3.83
  p-value = 2 * P(Z > 3.83) = 0.00013
  0.00013 < 0.05  -->  REJECT H0.
  The class mean is significantly different from 65.
""")

---
## 🤖 Part D — AI-Augmented Task

**Prompt used:**
> *Explain normal distribution, Z-score, and hypothesis testing with a simple Python example.*

In [ ]:
print("""
AI OUTPUT SUMMARY:
  The AI explained all three concepts with a student exam score example.
  - Normal distribution: described as bell curve, parameterised by mean/std,
    showed a histogram + PDF curve plot.
  - Z-score: formula Z = (x-mean)/std, shown applied to a list of scores.
  - Hypothesis testing: H0: class mean = 60, used scipy.stats.ttest_1samp,
    interpreted p-value against alpha = 0.05.
""")

In [ ]:
# Reproduce and verify the AI's code
np.random.seed(5)
exam_scores = list(np.random.normal(loc=68, scale=12, size=50).round(1))

# AI's Z-score application
mean_scores = np.mean(exam_scores)
std_scores  = np.std(exam_scores)
z_scores    = [(x - mean_scores) / std_scores for x in exam_scores]

print(f"Exam scores mean: {mean_scores:.2f}  std: {std_scores:.2f}")
print(f"First 5 Z-scores: {[round(z,3) for z in z_scores[:5]]}")
print()

# AI's hypothesis test: H0 mean = 60
t_stat, p_val = stats.ttest_1samp(exam_scores, popmean=60)
print(f"H0: class mean = 60")
print(f"t-stat = {t_stat:.4f},  p-value = {p_val:.6f}")
print("REJECT H0" if p_val < 0.05 else "FAIL TO REJECT H0")
print("(p < 0.05 -- class mean is significantly above 60)")

# Plot the AI's suggested histogram
fig, ax = plt.subplots()
ax.hist(exam_scores, bins=20, density=True, color="steelblue", alpha=0.7, edgecolor="white")
x_r = np.linspace(min(exam_scores), max(exam_scores), 300)
ax.plot(x_r, stats.norm.pdf(x_r, mean_scores, std_scores), color="red", lw=2)
ax.axvline(60, color="orange", linestyle="--", lw=2, label="H0 mean = 60")
ax.axvline(mean_scores, color="green", linestyle="--", lw=2, label=f"Sample mean = {mean_scores:.1f}")
ax.set_title("Exam Scores with H0 and Sample Mean Marked")
ax.set_xlabel("Score")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

### Evaluation

**Is the explanation correct?**  
Yes — the AI's description of all three concepts was accurate. The Z-score formula and hypothesis test logic were both correct.

**Is the code logically correct and runnable?**  
The code ran without errors (verified above). One issue: the AI used `stats.ttest_1samp` without first checking whether the sample is large enough for the t-distribution to approximate the Z-test well (n=50 is fine here, but the distinction wasn't mentioned). For very small samples, the t-test is preferred; for n ≥ 30, the Z-test and t-test give nearly identical results.

**What I would add / improve:**
- The AI did not demonstrate the false positive rate simulation (Part A5) — which is the most intuitive way to understand what alpha *actually means* in practice rather than as a theoretical threshold
- The AI's Z-score list comprehension used a lambda inside without explanation; for beginners a plain `for` loop is clearer
- The AI did not add the H0 mean line to the histogram (added above), which makes it visually obvious why we reject H0 when the sample mean is far from it
